This is the first chapter in the implementation-dedicated part of the book. 

For some reason, the pdf I have available cuts off the last ~5 pages of this chapter, so I couldn't do the space bar implementation (21.3)

In [16]:
#Section 21.1. Plane bar element
import numpy as np

def PlaneBar2Stiffness(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS: computes stiffness matrix
    OUTPUT: Ke stiffness matrix
    """

    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1
    
    Em, Gm, rho, alpha = mprop
    (A,) = fprop

    LL = x21**2+y21**2 
    L = np.sqrt(LL)
    LLL = LL*L

    Ke = (Em*A/LLL) * np.array([
        [ x21*x21,  x21*y21, -x21*x21, -x21*y21],
        [ y21*x21,  y21*y21, -y21*x21, -y21*y21],
        [-x21*x21, -x21*y21,  x21*x21,  x21*y21],
        [-y21*x21, -y21*y21,  y21*x21,  y21*y21]
    ])

    return Ke


def PlaneBar2ConsMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS: computes mass matrix
    OUTPUT: MeC mass matrix
    """
    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop
    (A,) = fprop

    L = np.sqrt(x21**2 + y21**2)
    MeC = (rho*A*L/6) * np.array([
        [2, 0, 1, 0], [0,2,0,1], [1,0,2,0], [0, 1, 0, 2]
    ])
    return MeC


def PlaneBar2LumpMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, input as list), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS: computes lumped mass matrix
    OUTPUT: MeL lumped mass matrix
    """
    (x1, y1), (x2, y2) = ncoor
    
    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop
    (A,) = fprop
    
    L = np.sqrt(x21**2 + y21**2)

    MeL = (rho * A * L / 2) * np.identity(4)
    return MeL

In [20]:
# Testing of plane bar element
ncoor = [[0, 0], [30, 40]]
mprop = [1000, 0, 0.6, 0]
fprop = [5]

Ke = PlaneBar2Stiffness(ncoor, mprop, fprop)
print("Numerical Elem Stiff Matrix:")
print(Ke)
eigvals = np.linalg.eigvalsh(Ke)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of Ke =", eigvals)
print("\nSymmetry check=", np.allclose(Ke,Ke.T))
print()

MeC = PlaneBar2ConsMass(ncoor, mprop, fprop)
print("Numerical consistent Mass Matrix: ")
print(MeC)
eigvals = np.linalg.eigvalsh(MeC)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeC =", eigvals)
print("Sympetry check=", np.allclose(MeC,MeC.T))
print()

MeL = PlaneBar2LumpMass(ncoor, mprop, fprop)
print("Numerical Lumped Mass Matrix:")
print(MeL)
eigvals = np.linalg.eigvalsh(MeL)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeL", eigvals)

Numerical Elem Stiff Matrix:
[[ 36.  48. -36. -48.]
 [ 48.  64. -48. -64.]
 [-36. -48.  36.  48.]
 [-48. -64.  48.  64.]]
Eigenvalues of Ke = [  0.   0.   0. 200.]

Symmetry check= True

Numerical consistent Mass Matrix: 
[[50.  0. 25.  0.]
 [ 0. 50.  0. 25.]
 [25.  0. 50.  0.]
 [ 0. 25.  0. 50.]]
Eigenvalues of MeC = [25. 25. 75. 75.]
Sympetry check= True

Numerical Lumped Mass Matrix:
[[75.  0.  0.  0.]
 [ 0. 75.  0.  0.]
 [ 0.  0. 75.  0.]
 [ 0.  0.  0. 75.]]
Eigenvalues of MeL [75. 75. 75. 75.]


In [18]:
# Section 21.2 Plane beam-column element
import numpy as np

def PlaneBeamColumn2Stiffness(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, Izz), 
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic. 
    PROCESS: computes stiffness matrix out of individual K for axial and bending components.
    OUTPUT: plane beam column stiffness matrix
    """

    (x1, y1), (x2, y2) = ncoor

    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop

    A, Izz = fprop

    LL = x21**2 + y21**2

    L = np.sqrt(LL)

    LLL = LL * L

    c = x21 / L #cosine
    s = y21 / L # sine

    #factors for bar and beam stiffnesses
    ra = Em*A/L
    rb =  2 * Em * Izz / LLL

    # local system stiffness matrix
    Kebar = ra * np.array([
        [1, 0, 0, -1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [-1, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]
    ], dtype=float) + rb * np.array([
        [0, 0, 0, 0, 0, 0],
        [0, 6, 3*L, 0, -6, 3*L],
        [0, 3*L, 2*LL, 0, -3*L, LL],
        [0, 0, 0, 0, 0, 0],
        [0, -6, -3*L, 0, 6, -3*L],
        [0, 3*L, LL, 0, -3*L, 2*LL]
    ], dtype=float)

    T = np.array([
        [c, s, 0, 0, 0, 0],
        [-s, c, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, c, s, 0 ],
        [0, 0, 0, -s, c, 0],
        [0, 0, 0, 0, 0, 1]
    ], dtype=float)

    # global stiffness matrix

    Ke = T.T @ Kebar @ T
    return Ke

    


In [41]:
# Testing of beam-column element

ncoor = [[0, 0], [3, 4]]
mprop = [100, 0, 84/5, 0]
fprop = [125, 250]

Ke = PlaneBeamColumn2Stiffness(ncoor, mprop, fprop)
print("Numerical Elem Stiff Matrix: ")
print(Ke)
eigvals = np.linalg.eigvalsh(Ke)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of Ke =", eigvals)

Numerical Elem Stiff Matrix: 
[[ 2436.    48. -4800. -2436.   -48. -4800.]
 [   48.  2464.  3600.   -48. -2464.  3600.]
 [-4800.  3600. 20000.  4800. -3600. 10000.]
 [-2436.   -48.  4800.  2436.    48.  4800.]
 [  -48. -2464. -3600.    48.  2464. -3600.]
 [-4800.  3600. 10000.  4800. -3600. 20000.]]
Eigenvalues of Ke = [    0.     0.     0.  5000. 10000. 34800.]


In [42]:
# Section 21.2, mass matrix and lumped mass matrix

def PlaneBeamColumn2ConsMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, Izz),
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic.
    PROCESS: computes mass matrix out of individual axial and bending stiffness matrices.
    OUTPUT: plane beam-column consistent mass matrix
    """

    (x1, y1), (x2, y2) = ncoor

    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop

    A, Izz = fprop

    LL = x21**2 + y21**2

    L = np.sqrt(LL)

    c = x21 / L  # cosine
    s = y21 / L  # sine

    # element mass
    m = rho * L * A

    # local system consistent mass matrix
    MeClocal = (m / 6) * np.array([
        [2, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [1, 0, 0, 2, 0, 0],
        [0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0]
    ], dtype=float) + (m / 420) * np.array([
        [0, 0, 0, 0, 0, 0],
        [0, 156, 22 * L, 0, 54, -13 * L],
        [0, 22 * L, 4 * LL, 0, 13 * L, -3 * LL],
        [0, 0, 0, 0, 0, 0],
        [0, 54, 13 * L, 0, 156, -22 * L],
        [0, -13 * L, -3 * LL, 0, -22 * L, 4 * LL]
    ], dtype=float)

    T = np.array([
        [c, s, 0, 0, 0, 0],
        [-s, c, 0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0],
        [0, 0, 0, c, s, 0],
        [0, 0, 0, -s, c, 0],
        [0, 0, 0, 0, 0, 1]
    ], dtype=float)

    # global consistent mass matrix
    MeC = T.T @ MeClocal @ T

    return MeC


def PlaneBeamColumn2LumpMass(ncoor, mprop, fprop):
    """
    INPUT: ncoor (array [[x1, y1], [x2,y2]]), mprop (E, G, rho, alpha), fprop (Area, Izz),
        SIDENOTE: original Felippa code includes "numer" as option for numerical computation instead of symbolic.
    PROCESS: computes lumped mass matrix
    OUTPUT: plane beam-column lumped mass matrix 
    """

    (x1, y1), (x2, y2) = ncoor

    x21 = x2 - x1
    y21 = y2 - y1

    Em, Gm, rho, alpha = mprop

    A, Izz = fprop

    LL = x21**2 + y21**2

    L = np.sqrt(LL)

    # HRZ lumping scheme for rotational masses
    MeL = (rho * A * L / 2) * np.array([
        [1, 0, 0, 0, 0, 0],
        [0, 1, 0, 0, 0, 0],
        [0, 0, LL / 39, 0, 0, 0],
        [0, 0, 0, 1, 0, 0],
        [0, 0, 0, 0, 1, 0],
        [0, 0, 0, 0, 0, LL / 39]
    ], dtype=float)

    return MeL


In [44]:
# Testing of beam-column element

ncoor = [[0, 0], [3, 4]]
mprop = [100, 0, 84/5, 0]
fprop = [125, 250]

MeC = PlaneBeamColumn2ConsMass(ncoor, mprop, fprop)
print("Numerical Consistent Mass Matrix: ")
print(MeC)
eigvals = np.linalg.eigvalsh(MeC)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeC=", eigvals)
# Squared frequencies
omega2 = np.linalg.eigvals(np.linalg.inv(MeC) @ Ke)
omega2 = np.real_if_close(omega2)
omega2 = np.where(np.abs(omega2) < 1e-7, 0.0, omega2)
print("Squared frequencies (consistent) =", omega2)

MeL = PlaneBeamColumn2LumpMass(ncoor, mprop, fprop)
print("Numerical Lumped Mass Matrix:")
with np.printoptions(linewidth=120):
    print(MeL)
eigvals = np.linalg.eigvalsh(MeL)
eigvals = np.where(np.abs(eigvals) < 1e-10, 0.0, eigvals)
print("Eigenvalues of MeL=", eigvals)
# Squared frequencies 
omega2 = np.linalg.eigvals(np.linalg.inv(MeL) @ Ke)
omega2 = np.real_if_close(omega2)
omega2 = np.where(np.abs(omega2) < 1e-7, 0.0, omega2)
print("Squared frequencies (lumped) =", omega2)

Numerical Consistent Mass Matrix: 
[[ 3756.  -192. -2200.  1494.   192.  1300.]
 [ -192.  3644.  1650.   192.  1606.  -975.]
 [-2200.  1650.  2500. -1300.   975. -1875.]
 [ 1494.   192. -1300.  3756.  -192.  2200.]
 [  192.  1606.   975.  -192.  3644. -1650.]
 [ 1300.  -975. -1875.  2200. -1650.  2500.]]
Eigenvalues of MeC= [ 106.95  415.68 1750.   3068.05 5250.   9209.32]
Squared frequencies (consistent) = [160.     2.86  13.71   0.     0.     0.  ]
Numerical Lumped Mass Matrix:
[[5250.      0.      0.      0.      0.      0.  ]
 [   0.   5250.      0.      0.      0.      0.  ]
 [   0.      0.   3365.38    0.      0.      0.  ]
 [   0.      0.      0.   5250.      0.      0.  ]
 [   0.      0.      0.      0.   5250.      0.  ]
 [   0.      0.      0.      0.      0.   3365.38]]
Eigenvalues of MeL= [3365.38 3365.38 5250.   5250.   5250.   5250.  ]
Squared frequencies (lumped) = [9.83 0.   0.95 2.97 0.   0.  ]
